# iGEMerDB production data quality audit

This notebook audits the exact public export used by the production build. It reports structural integrity separately from historical coverage; a zero integrity error count does not imply that every historical award exists upstream.

In [1]:
from collections import Counter
from pathlib import Path
import hashlib, json

root = Path.cwd()
if not (root / 'public/data/igem.json').exists():
    root = root.parent
source_path = root / 'public/data/igem.json'
raw = source_path.read_bytes()
data = json.loads(raw)
print({'path': str(source_path), 'bytes': len(raw), 'sha256': hashlib.sha256(raw).hexdigest().upper(), 'generated_at': data['meta']['generated_at']})

{'path': 'D:\\Windows\\Documents\\ACoding\\iGEMerDB\\OIerDb\\public\\data\\igem.json', 'bytes': 42619593, 'sha256': '54AF06530E456D58CC868379CE92F0DC3E5A4836D69CBC62E51ABC06E9355D37', 'generated_at': '2026-07-22T12:37:01.042603+00:00'}


In [2]:
teams = data['teams']; members = data['members']; roster = data['roster']; awards = data['awards']; team_awards = data['team_awards']
team_ids = {row['id'] for row in teams}; member_ids = {row['uuid'] for row in members}; award_ids = {row['uuid'] for row in awards}
def duplicate_count(values):
    counts = Counter(values)
    return sum(count - 1 for count in counts.values() if count > 1)
integrity = {
    'duplicate_team_ids': duplicate_count(row['id'] for row in teams),
    'duplicate_member_uuids': duplicate_count(row['uuid'] for row in members),
    'duplicate_roster_edges': duplicate_count((row['team_id'], row['member_uuid']) for row in roster),
    'orphan_roster_teams': sum(row['team_id'] not in team_ids for row in roster),
    'orphan_roster_members': sum(row['member_uuid'] not in member_ids for row in roster),
    'orphan_team_award_teams': sum(row['team_id'] not in team_ids for row in team_awards),
    'orphan_team_award_definitions': sum(row['award_uuid'] not in award_ids for row in team_awards),
}
print(integrity)
assert not any(integrity.values()), integrity

{'duplicate_team_ids': 0, 'duplicate_member_uuids': 0, 'duplicate_roster_edges': 0, 'orphan_roster_teams': 0, 'orphan_roster_members': 0, 'orphan_team_award_teams': 0, 'orphan_team_award_definitions': 0}


In [3]:
status_counts = Counter(row['export_category'] for row in teams)
visible = [row for row in teams if row['default_visible']]
role_counts = Counter(row['role_inferred'] for row in roster)
unsupported_degree_roles = [row for row in roster if row['role_inferred'] in {'Undergrad', 'Graduate'} and not (row.get('snapshot_title') or '').strip()]
policy = {
    'raw_teams': len(teams), 'default_visible_teams': len(visible),
    'visible_non_accepted': sum(row['export_category'] != 'accepted' for row in visible),
    'status_counts': dict(sorted(status_counts.items())),
    'role_counts': dict(sorted(role_counts.items())),
    'unsupported_degree_roles': len(unsupported_degree_roles),
}
print(json.dumps(policy, indent=2))
assert policy['visible_non_accepted'] == 0
assert policy['unsupported_degree_roles'] == 0

{
  "raw_teams": 5518,
  "default_visible_teams": 5283,
  "visible_non_accepted": 0,
  "status_counts": {
    "accepted": 5283,
    "demo-test": 19,
    "disqualified": 28,
    "withdrawn": 188
  },
  "role_counts": {
    "Advisor": 16836,
    "Graduate": 781,
    "Other": 715,
    "PI": 9547,
    "Student": 75607,
    "Undergrad": 1335
  },
  "unsupported_degree_roles": 0
}


In [4]:
coverage = []
for row in data['meta']['coverage']:
    denominator = row['default_visible_count']
    coverage.append({
        'year': row['year'],
        'visible_teams': denominator,
        'teams_with_awards': row['teams_with_awards'],
        'award_team_coverage_pct': round(100 * row['teams_with_awards'] / denominator, 1) if denominator else 0.0,
        'fetch_errors': row['fetch_error_count'],
    })
print(json.dumps(coverage, indent=2))
print('Historical coverage warning:', [row for row in coverage if row['year'] <= 2007])

[
  {
    "year": 2004,
    "visible_teams": 5,
    "teams_with_awards": 0,
    "award_team_coverage_pct": 0.0,
    "fetch_errors": 0
  },
  {
    "year": 2005,
    "visible_teams": 13,
    "teams_with_awards": 0,
    "award_team_coverage_pct": 0.0,
    "fetch_errors": 0
  },
  {
    "year": 2006,
    "visible_teams": 41,
    "teams_with_awards": 3,
    "award_team_coverage_pct": 7.3,
    "fetch_errors": 0
  },
  {
    "year": 2007,
    "visible_teams": 53,
    "teams_with_awards": 1,
    "award_team_coverage_pct": 1.9,
    "fetch_errors": 0
  },
  {
    "year": 2008,
    "visible_teams": 77,
    "teams_with_awards": 67,
    "award_team_coverage_pct": 87.0,
    "fetch_errors": 0
  },
  {
    "year": 2009,
    "visible_teams": 102,
    "teams_with_awards": 90,
    "award_team_coverage_pct": 88.2,
    "fetch_errors": 0
  },
  {
    "year": 2010,
    "visible_teams": 117,
    "teams_with_awards": 105,
    "award_team_coverage_pct": 89.7,
    "fetch_errors": 0
  },
  {
    "year": 2011,
  